# Bài 6(a) — Tập phổ biến bằng Apriori

**Võ Thiện Linh — MSSV: 6451071041**

Dùng Groceries Dataset (`groceries.csv`) với `min_sup` từ 0,01 đến 0,1 theo tỉ lệ (1%–10%), tăng 0,01. Thời gian là trung vị của ba lần chạy; không tính thời gian đọc dữ liệu.

## Đọc và kiểm tra dữ liệu

In [ ]:
import csv
import math
import time
from collections import Counter
from itertools import combinations
from pathlib import Path
from statistics import median

import pandas as pd
from IPython.display import display

data_candidates = [Path('groceries.csv'), Path('6451071041_LAB1_Bai6_Trang45') / 'groceries.csv', Path('LAB1') / 'Phan02' / '6451071041_LAB1_Bai6_Trang45' / 'groceries.csv']
data_path = next((path for path in data_candidates if path.is_file()), None)
if data_path is None:
    raise FileNotFoundError(f'Không tìm thấy groceries.csv. Thư mục đang chạy: {Path.cwd()}')
with data_path.open(encoding='utf-8', newline='') as csv_file:
    transactions = [frozenset(item.strip() for item in row if item.strip()) for row in csv.reader(csv_file)]
transaction_count = len(transactions)
item_count = len(set().union(*transactions))
print(f'Đã đọc: {data_path.resolve()}')
print(f'Số giao dịch: {transaction_count:,}; số mặt hàng: {item_count:,}')
assert (transaction_count, item_count) == (9835, 169)
support_levels = [percent / 100 for percent in range(1, 11)]
repeat_count = 3

## Thuật toán Apriori
Sinh ứng viên theo từng kích thước và chỉ giữ ứng viên có mọi tập con kích thước nhỏ hơn một phần tử là phổ biến. Ngưỡng đếm tối thiểu được tính bằng `ceil(min_sup × số giao dịch)`. 

In [ ]:
def apriori(transactions, min_count):
    item_counts = Counter(item for transaction in transactions for item in transaction)
    previous = {frozenset([item]): count for item, count in item_counts.items() if count >= min_count}
    all_frequent = dict(previous)
    frequent_singletons = set(item_counts)
    itemset_size = 2
    while previous:
        previous_tuples = sorted(tuple(sorted(itemset)) for itemset in previous)
        previous_set = set(previous)
        candidates = set()
        for left, right in combinations(previous_tuples, 2):
            if left[:-1] != right[:-1]:
                continue
            candidate = frozenset(left + (right[-1],))
            if len(candidate) == itemset_size and all(frozenset(subset) in previous_set for subset in combinations(candidate, itemset_size - 1)):
                candidates.add(candidate)
        if not candidates:
            break
        candidate_counts = Counter()
        for transaction in transactions:
            active_items = sorted(transaction & frequent_singletons)
            if len(active_items) >= itemset_size:
                for parts in combinations(active_items, itemset_size):
                    itemset = frozenset(parts)
                    if itemset in candidates:
                        candidate_counts[itemset] += 1
        previous = {itemset: count for itemset, count in candidate_counts.items() if count >= min_count}
        all_frequent.update(previous)
        itemset_size += 1
    return all_frequent

apriori_results = {}
rows = []
for min_sup in support_levels:
    min_count = math.ceil(min_sup * transaction_count)
    elapsed = []
    for _ in range(repeat_count):
        start = time.perf_counter()
        frequent = apriori(transactions, min_count)
        elapsed.append(time.perf_counter() - start)
    apriori_results[min_sup] = frequent
    rows.append({'min_sup': f'{min_sup:.0%}', 'min_count': min_count, 'Số tập phổ biến': len(frequent), 'Thời gian Apriori (giây)': median(elapsed)})
apriori_summary = pd.DataFrame(rows)
print('Kết quả và thời gian thực hiện (trung vị 3 lần):')
display(apriori_summary)
frequent_itemsets = pd.DataFrame([{'min_sup': f'{s:.0%}', 'tập mặt hàng': ', '.join(sorted(items)), 'kích thước tập': len(items), 'số giao dịch hỗ trợ': count, 'support thực tế': count / transaction_count} for s, results in apriori_results.items() for items, count in sorted(results.items(), key=lambda pair: (len(pair[0]), tuple(sorted(pair[0]))))])
print(f'Tổng số dòng tập phổ biến qua các ngưỡng: {len(frequent_itemsets):,}')
display(frequent_itemsets)